# 02 误差修正模型 (ECM) 配对交易

## 论文参考
- **作者**: Yourui Wang
- **年份**: 2023
- **标题**: *Pairs Trading Strategy Based on Cointegration Model and Error Correction Model (ECM)*
- **关键结果**: Sharpe Ratio ~1.5
- **摘要**: 本文使用误差修正模型(ECM)改进传统协整配对交易。ECM不仅捕捉长期均衡关系，
  还建模短期动态调整过程，使价差预测更准确，信号更及时。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### 1. 协整与ECM
Granger表示定理: 若 $P_A, P_B$ 协整，则存在ECM:
$$\Delta P_A(t) = \alpha_1 \cdot e_{t-1} + \sum_{i=1}^p \gamma_i \Delta P_A(t-i) + \sum_{j=1}^q \delta_j \Delta P_B(t-j) + \varepsilon_t$$

其中:
- $e_{t-1} = P_A(t-1) - \beta P_B(t-1) - \mu$: 协整残差(误差修正项)
- $\alpha_1$: 误差修正系数 (应为负数，表示向均衡回归)
- $\gamma_i, \delta_j$: 短期动态系数

### 2. ECM交易策略
- 使用ECM预测下一期价差变化 $\hat{\Delta}e_t$
- 当预测价差将收缩 → 做多当前偏大的价差
- 当预测价差将扩张 → 平仓或反向

### 3. 信号生成
$$\text{ECM\_signal}_t = \alpha_1 \cdot e_{t-1} + \text{short-term dynamics}$$
- $\text{signal} > \text{threshold}$: 预测A将上涨 → 买A卖B
- $\text{signal} < -\text{threshold}$: 预测A将下跌 → 卖A买B

In [ ]:
# === Cell 3: ECM价差均衡修正动画 ===
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# Simulate ECM spread dynamics
T = 300
alpha = -0.15  # Error correction speed
equilibrium = 0.0
spread = np.zeros(T)
spread[0] = 2.0  # Start away from equilibrium

for t in range(1, T):
    ecm_correction = alpha * (spread[t-1] - equilibrium)
    noise = np.random.randn() * 0.3
    spread[t] = spread[t-1] + ecm_correction + noise

frames = []
for step in range(20, T, 5):
    ecm_forces = [alpha * (spread[i] - equilibrium) for i in range(step)]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=list(range(step)), y=spread[:step], mode='lines',
                       name='价差', line=dict(color='#2196F3', width=2)),
            go.Scatter(x=list(range(step)), y=[equilibrium]*step, mode='lines',
                       name='均衡值', line=dict(color='gray', width=1.5, dash='dash')),
            go.Bar(x=list(range(step)), y=ecm_forces, name='ECM修正力',
                   marker_color=['#4CAF50' if f < 0 else '#F44336' for f in ecm_forces],
                   opacity=0.4),
        ],
        name=f'Step {step}'
    ))

fig = go.Figure(data=frames[0].data, frames=frames)
fig.update_layout(
    title=dict(text='ECM: 价差围绕均衡值振荡 + 误差修正力'),
    xaxis_title='时间', yaxis_title='价差 / ECM力',
    height=500, width=1000,
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='▶ 播放', method='animate',
             args=[None, {'frame': {'duration': 150}}]),
    ])],
    sliders=[dict(
        steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames],
    )],
)
fig.show()

In [ ]:
# === Cell 4: 导入与配置 ===
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from sklearn.linear_model import LinearRegression

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.backtest_engine import PairsBacktester
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table)
from shared.constants import *

print('All imports successful.')

In [ ]:
# === Cell 5: 数据获取 ===
stock_a = get_stock_daily(STOCK_ICBC, DEFAULT_START, DEFAULT_END)
stock_b = get_stock_daily(STOCK_CCB, DEFAULT_START, DEFAULT_END)
benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)

price_a = stock_a['close']
price_b = stock_b['close']

common_idx = price_a.index.intersection(price_b.index)
price_a = price_a.loc[common_idx]
price_b = price_b.loc[common_idx]

print(f'ICBC: {len(price_a)} days, CCB: {len(price_b)} days')

In [ ]:
# === Cell 6: ECM模型估计 ===

# Step 1: Cointegration regression
ols_beta = np.polyfit(price_b.values, price_a.values, 1)
beta, intercept = ols_beta[0], ols_beta[1]
residual = price_a - beta * price_b - intercept

# Cointegration test
coint_stat, coint_p, _ = coint(price_a, price_b)
print(f'Cointegration p-value: {coint_p:.6f} (Cointegrated: {coint_p < 0.05})')

# Step 2: ECM estimation
# Delta P_A(t) = alpha * e(t-1) + gamma * Delta P_A(t-1) + delta * Delta P_B(t-1) + eps
delta_pa = price_a.diff().dropna()
delta_pb = price_b.diff().dropna()
e_lag = residual.shift(1).dropna()

# Align all
common = delta_pa.index.intersection(delta_pb.index).intersection(e_lag.index)
delta_pa = delta_pa.loc[common]
delta_pb = delta_pb.loc[common]
e_lag = e_lag.loc[common]

# Add lagged differences
delta_pa_lag1 = delta_pa.shift(1).fillna(0)
delta_pb_lag1 = delta_pb.shift(1).fillna(0)

X_ecm = pd.DataFrame({
    'e_lag': e_lag.values,
    'delta_pa_lag1': delta_pa_lag1.values,
    'delta_pb_lag1': delta_pb_lag1.values,
}, index=common)

ecm_model = LinearRegression()
ecm_model.fit(X_ecm.values, delta_pa.values)

alpha_ecm = ecm_model.coef_[0]
gamma_ecm = ecm_model.coef_[1]
delta_ecm = ecm_model.coef_[2]

print(f'\n=== ECM Coefficients ===')
print(f'  Error correction (alpha): {alpha_ecm:.4f} (should be negative)')
print(f'  Delta_PA lag1 (gamma): {gamma_ecm:.4f}')
print(f'  Delta_PB lag1 (delta): {delta_ecm:.4f}')
print(f'  R-squared: {ecm_model.score(X_ecm.values, delta_pa.values):.4f}')

In [ ]:
# === Cell 7: ECM信号生成 + 回测 ===

# ECM-predicted change in spread
ecm_prediction = ecm_model.predict(X_ecm.values)
ecm_pred_series = pd.Series(ecm_prediction, index=common, name='ecm_pred')

# Also compute Z-score of residual for entry/exit
lookback = 60
spread_mean = residual.rolling(lookback).mean()
spread_std = residual.rolling(lookback).std()
z_score = (residual - spread_mean) / (spread_std + 1e-8)
z_score = z_score.reindex(common)

# Combine Z-score with ECM prediction for signals
ENTRY_Z = 1.5
EXIT_Z = 0.3

position = pd.Series(0.0, index=common)
current_pos = 0

for i in range(lookback, len(common)):
    z = z_score.iloc[i]
    ecm_pred = ecm_pred_series.iloc[i]
    if np.isnan(z):
        position.iloc[i] = current_pos
        continue

    if current_pos == 0:
        # Enter when Z-score is extreme AND ECM predicts reversion
        if z > ENTRY_Z and ecm_pred < 0:  # Spread too high, ECM says it will shrink
            current_pos = -1  # Short spread
        elif z < -ENTRY_Z and ecm_pred > 0:  # Spread too low, ECM says it will grow
            current_pos = 1   # Long spread
    elif current_pos == 1:
        if z > -EXIT_Z:  # Spread reverted
            current_pos = 0
    elif current_pos == -1:
        if z < EXIT_Z:
            current_pos = 0

    position.iloc[i] = current_pos

# Compute returns
ret_a = price_a.pct_change().fillna(0).reindex(common)
ret_b = price_b.pct_change().fillna(0).reindex(common)
spread_returns = position.shift(1).fillna(0) * (ret_a - beta * ret_b)

pos_change = position.diff().fillna(0)
trade_cost = pos_change.abs() * (COMMISSION_RATE * 2 + STAMP_TAX_RATE)
strategy_returns = spread_returns - trade_cost
equity = INITIAL_CAPITAL * (1 + strategy_returns).cumprod()

from shared.metrics import summary_table
bench_ret = benchmark['close'].reindex(common).ffill().pct_change().fillna(0) if 'close' in benchmark.columns else None
metrics = summary_table(strategy_returns, equity, bench_ret)

print('=== ECM Pairs Trading Results ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 8: 对比实验 (ECM vs 纯Z-Score) ===

# Pure Z-score strategy (without ECM confirmation)
bt_pure = PairsBacktester(
    initial_capital=INITIAL_CAPITAL,
    entry_z=ENTRY_Z, exit_z=EXIT_Z
)
result_pure = bt_pure.run(price_a, price_b, hedge_ratio=beta,
                          benchmark_prices=benchmark.get('close'))

print('=== Pure Z-Score (No ECM) ===')
for k, v in result_pure['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# === Cell 9: 可视化 ===

# 1. Spread + ECM prediction
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

ax1 = axes[0]
ax1.plot(residual.index, residual, 'b-', linewidth=0.8, label='协整残差(价差)')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('协整残差(价差)序列', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(ecm_pred_series.index, ecm_pred_series, 'orange', linewidth=0.8, label='ECM预测价差变化')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('ECM 预测的价差变化方向', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
ax3.plot(z_score.index, z_score, 'k-', linewidth=0.8)
ax3.axhline(y=ENTRY_Z, color='red', linestyle='--', alpha=0.7, label=f'Entry +/-{ENTRY_Z}')
ax3.axhline(y=-ENTRY_Z, color='green', linestyle='--', alpha=0.7)
ax3.fill_between(z_score.index, -EXIT_Z, EXIT_Z, alpha=0.1, color='green', label='平仓区')
long_m = position > 0
short_m = position < 0
ax3.fill_between(position.index, z_score.min(), z_score.max(),
                  where=long_m, alpha=0.15, color='green', label='做多')
ax3.fill_between(position.index, z_score.min(), z_score.max(),
                  where=short_m, alpha=0.15, color='red', label='做空')
ax3.set_title('Z-Score + ECM交易信号', fontsize=13, fontweight='bold')
ax3.legend(fontsize=9, ncol=3)
ax3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Equity comparison
from shared.visualization import plot_cumulative_comparison
pure_eq = result_pure['equity_curve'] / result_pure['equity_curve'].iloc[0]
ecm_eq = equity / equity.iloc[0]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ecm_eq.index, ecm_eq, 'b-', linewidth=2, label='ECM配对交易')
ax.plot(pure_eq.index, pure_eq, 'r--', linewidth=1.5, label='纯Z-Score配对交易')
ax.set_title('ECM vs 纯Z-Score 配对交易对比', fontsize=14, fontweight='bold')
ax.set_xlabel('日期')
ax.set_ylabel('净值')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Drawdown & Metrics
plot_drawdown(equity, title='ECM配对交易 - 回撤')
plot_metrics_table(metrics, title='ECM配对交易 - 绩效指标')

## 结果讨论

### 核心发现
1. **ECM修正项**: 误差修正系数 $\alpha < 0$ 确认价差存在向均衡回归的趋势
2. **信号过滤**: ECM预测方向与Z-score方向一致时入场，减少假信号
3. **短期动态**: ECM捕捉的滞后价格变化提供额外的短期预测信息

### ECM vs 纯Z-Score
- ECM增加了方向确认，可能减少交易次数但提高信号质量
- 纯Z-Score仅依赖统计极端值，无方向性预测

### 与论文对比
- Wang (2023) 报告Sharpe约1.5，使用更长的历史数据和月度调仓
- 本实现为日频ECM信号，交易更活跃

### 改进方向
- VECM (向量误差修正): 同时对两只股票建模
- 时变ECM系数 (滚动窗口重新估计)
- 加入成交量和波动率作为ECM额外变量